# 🐍 Master Guide: Behavioral Design Patterns in Python
> **Single-Source Revision Guide**: Covers Observer, Strategy, Command (with Undo/Redo), State Machine, and Chain of Responsibility patterns.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Behavioral patterns focus on algorithms, request flows, and the assignment of responsibilities between cooperating objects, ensuring low coupling and high flexibility.

---
## 🟢 2. Part 1: Beginner Fundamentals

### Step 1: The Observer Pattern (Pub-Sub Event Bus)
Notifies interested observers automatically when state changes occur.

In [ ]:
class NewsletterPublisher:
    def __init__(self):
        self._subscribers = set()
    
    def subscribe(self, observer_fn):
        self._subscribers.add(observer_fn)
    
    def unsubscribe(self, observer_fn):
        self._subscribers.discard(observer_fn)
    
    def broadcast(self, news_article):
        print(f"\nBroadcasting: '{news_article}' to {len(self._subscribers)} subscriber(s):")
        for sub in self._subscribers:
            sub(news_article)

# Observers (can simply be functions in Python!):
def email_subscriber(news): print(" [Email] Sending to inbox:", news)
def push_subscriber(news):  print(" [Push] Alert on phone  :", news)

pub = NewsletterPublisher()
pub.subscribe(email_subscriber)
pub.subscribe(push_subscriber)

pub.broadcast("Python 3.14 Released with Massive Optimizations!")

---
## 🟡 3. Part 2: Intermediate Features

### Step 2: The Strategy Pattern (Pluggable Algorithms)
In Python, first-class functions make the Strategy pattern completely trivial—just pass functions directly!

In [ ]:
# Pricing strategies:
def regular_pricing(price):
    return price

def holiday_discount_20(price):
    return price * 0.80

def vip_discount_30(price):
    return price * 0.70

class Order:
    def __init__(self, amount, strategy=regular_pricing):
        self.amount = amount
        self.strategy = strategy
    
    def total(self):
        return self.strategy(self.amount)

order = Order(100.0, strategy=holiday_discount_20)
print("Order with 20% discount : $", order.total())

order.strategy = vip_discount_30  # Swap strategy dynamically!
print("Order with VIP discount: $", order.total())

### Step 3: The Command Pattern (Encapsulated Actions with Undo/Rollback)
Encapsulates a request as an object, enabling parameterization of clients, queuing, and reversible operations.

In [ ]:
from abc import ABC, abstractmethod

class AccountReceiver:
    def __init__(self, owner, balance=1000.0):
        self.owner = owner
        self.balance = balance

class TransactionCommand(ABC):
    @abstractmethod
    def execute(self):
        pass

    @abstractmethod
    def undo(self):
        pass

class DepositCommand(TransactionCommand):
    def __init__(self, account: AccountReceiver, amount: float):
        self.account = account
        self.amount = amount

    def execute(self):
        self.account.balance += self.amount
        print(f"  [EXECUTED] Deposited Rs. {self.amount:,.2f} -> Balance: Rs. {self.account.balance:,.2f}")

    def undo(self):
        self.account.balance -= self.amount
        print(f"  [UNDONE/ROLLBACK] Reverted Deposit of Rs. {self.amount:,.2f} -> Balance: Rs. {self.account.balance:,.2f}")

class TransactionManager:
    def __init__(self):
        self._history = []

    def run_command(self, cmd: TransactionCommand):
        cmd.execute()
        self._history.append(cmd)

    def rollback_last(self):
        if not self._history:
            print("No transactions to rollback!")
            return
        last_cmd = self._history.pop()
        last_cmd.undo()

# Testing Command Pattern with Undo Rollback:
acc = AccountReceiver("Kavitha", balance=5000.0)
mgr = TransactionManager()

print("Initial Balance: Rs.", acc.balance)
mgr.run_command(DepositCommand(acc, 2000.0))
mgr.run_command(DepositCommand(acc, 3500.0))
print("\nTriggering rollback of last transaction:")
mgr.rollback_last()

---
## 📋 4. Part 3: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Pattern                            | Intent                                |
+------------------------------------+---------------------------------------+
| Observer                           | One-to-many state change broadcast   |
| Strategy                           | Swappable algorithms at runtime       |
| Command                            | Encapsulates action as object with undo|
| State                              | Alters behavior as internal state chgs|
| Chain of Responsibility            | Sequential handler request pipeline   |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Banking Event Notification Hub (Observer Pattern)
> **Core Principle**:
> - When an ATM transaction occurs, decoupled services (SMS alert, fraud detection, email summary) listen to the transaction event bus without tight coupling to the core ATM account engine.

In [ ]:
# Real-Time Event Hub (Observer Pattern)
class BankAccountEventHub:
    def __init__(self, account_no):
        self.account_no = account_no
        self.balance = 20000.0
        self._subscribers = []

    def attach_listener(self, callback):
        self._subscribers.append(callback)

    def _notify(self, event_type, amount):
        event = {
            "account": self.account_no,
            "type": event_type,
            "amount": amount,
            "current_balance": self.balance
        }
        for listener in self._subscribers:
            listener(event)

    def withdraw(self, amount):
        if amount > self.balance:
            self._notify("DECLINED_INSUFFICIENT_FUNDS", amount)
            return False
        self.balance -= amount
        self._notify("WITHDRAWAL_SUCCESS", amount)
        return True

# Decoupled Observers:
def sms_alert_service(event):
    print(f"  [SMS Gateway] Alert for {event['account']}: {event['type']} of Rs. {event['amount']:,.2f}")

def fraud_watchdog_service(event):
    if event["type"] == "DECLINED_INSUFFICIENT_FUNDS":
        print(f"  [FRAUD ALERT] High-risk declined transaction on {event['account']}!")

# Connect and test
hub = BankAccountEventHub("ACC-887711")
hub.attach_listener(sms_alert_service)
hub.attach_listener(fraud_watchdog_service)

print("Testing valid withdrawal:")
hub.withdraw(5000.0)

print("\nTesting overdrawing withdrawal:")
hub.withdraw(50000.0)

### 2. Payment Gateway Validation Strategy (Strategy Pattern)
Runtime strategy injection for credit cards, UPI IDs, and NetBanking validation rules.

In [ ]:
# Strategy Pattern Drill: Validation Engines
import re

def validate_upi(identifier: str) -> bool:
    return bool(re.match(r'^[a-zA-Z0-9._-]{2,49}@[a-zA-Z]{2,}$', identifier))

def validate_card(identifier: str) -> bool:
    clean = identifier.replace("-", "").replace(" ", "")
    return len(clean) == 16 and clean.isdigit()

class PaymentCheckout:
    def __init__(self, validation_strategy):
        self.validator = validation_strategy

    def process_payment(self, identifier: str, amount: float):
        if not self.validator(identifier):
            return f"[DECLINED] Invalid format for identifier '{identifier}'"
        return f"[SUCCESS] Approved Rs. {amount:,.2f} payment for '{identifier}'"

upi_checkout = PaymentCheckout(validation_strategy=validate_upi)
card_checkout = PaymentCheckout(validation_strategy=validate_card)

print(upi_checkout.process_payment("student@okicici", 12000.0))
print(card_checkout.process_payment("1234-5678-9876-5432", 25000.0))
print(upi_checkout.process_payment("invalid-upi", 500.0))

### 3. ATM Multi-Tier Currency Dispenser (Chain of Responsibility Pattern)
> **Core Principle**:
> - A withdrawal request passes down a chain of denomination handlers (`Rs. 2000` -> `Rs. 500` -> `Rs. 200` -> `Rs. 100`).
> - Each handler dispenses maximum notes for its denomination and passes the remainder to the next handler in the chain.

In [ ]:
# Chain of Responsibility: ATM Cash Dispenser
class DispenseHandler(ABC):
    def __init__(self, denomination: int):
        self.denomination = denomination
        self.next_handler = None

    def set_next(self, handler):
        self.next_handler = handler
        return handler

    def dispense(self, amount: int):
        num_notes = amount // self.denomination
        remainder = amount % self.denomination
        if num_notes > 0:
            print(f"  Dispensing {num_notes} x Rs. {self.denomination} note(s)")
        if remainder > 0:
            if self.next_handler:
                self.next_handler.dispense(remainder)
            else:
                print(f"  [ERROR] Cannot dispense remaining Rs. {remainder} (No matching lower denomination available)")

# Build the Chain: 2000 -> 500 -> 200 -> 100
h2000 = DispenseHandler(2000)
h500 = DispenseHandler(500)
h200 = DispenseHandler(200)
h100 = DispenseHandler(100)

h2000.set_next(h500).set_next(h200).set_next(h100)

print("--- ATM Dispensing Rs. 6,800 ---")
h2000.dispense(6800)

print("\n--- ATM Dispensing Rs. 3,750 ---")
h2000.dispense(3750)

### 4. ATM State Controller (State Pattern)
> **Core Principle**:
> - The behavior of an ATM depends dynamically on its current state (`NoCardState`, `CardInsertedState`, `AuthenticatedState`).
> - State transitions are handled cleanly by state objects rather than messy nested `if-elif-else` flags.

In [ ]:
# State Pattern: ATM State Controller
class ATMState(ABC):
    @abstractmethod
    def insert_card(self, context):
        pass
    @abstractmethod
    def enter_pin(self, context, pin):
        pass
    @abstractmethod
    def withdraw_cash(self, context, amount):
        pass

class NoCardState(ATMState):
    def insert_card(self, context):
        print("Card inserted. Please enter your 4-digit PIN.")
        context.set_state(context.card_inserted_state)
    def enter_pin(self, context, pin):
        print("Action Rejected: Please insert a debit card first!")
    def withdraw_cash(self, context, amount):
        print("Action Rejected: Please insert a debit card first!")

class CardInsertedState(ATMState):
    def insert_card(self, context):
        print("Action Rejected: A card is already inserted!")
    def enter_pin(self, context, pin):
        if pin == 1234:
            print("PIN Verified Successfully. Welcome!")
            context.set_state(context.authenticated_state)
        else:
            print("Invalid PIN entered. Please try again.")
    def withdraw_cash(self, context, amount):
        print("Action Rejected: Please authenticate with PIN first!")

class AuthenticatedState(ATMState):
    def insert_card(self, context):
        print("Action Rejected: Session active!")
    def enter_pin(self, context, pin):
        print("Already authenticated!")
    def withdraw_cash(self, context, amount):
        print(f"Dispensing Rs. {amount:,.2f}. Transaction complete. Ejecting card.")
        context.set_state(context.no_card_state)

class ATMContext:
    def __init__(self):
        self.no_card_state = NoCardState()
        self.card_inserted_state = CardInsertedState()
        self.authenticated_state = AuthenticatedState()
        self.state = self.no_card_state

    def set_state(self, state):
        self.state = state

    def insert_card(self): self.state.insert_card(self)
    def enter_pin(self, pin): self.state.enter_pin(self, pin)
    def withdraw_cash(self, amount): self.state.withdraw_cash(self, amount)

# Simulating ATM state progression:
atm_session = ATMContext()
print("--- ATM State Machine Test Run ---")
atm_session.enter_pin(1234)        # Blocked: No card
atm_session.insert_card()          # Transition to CardInsertedState
atm_session.withdraw_cash(1000)    # Blocked: Not authenticated yet
atm_session.enter_pin(1234)        # Transition to AuthenticatedState
atm_session.withdraw_cash(2500)    # Allowed! Dispenses & resets to NoCardState